In [1]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import wandb

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

In [2]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("WANDB_API_KEY")


In [3]:
wandb.login(key=secret_value_0)

wandb: Using wandb-core as the SDK backend. Please refer to https://wandb.me/wandb-core for more information.
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc


True

In [4]:
# set the wandb project where this run will be logged
os.environ["WANDB_PROJECT"]="MajorProject-try-1-multiclass"

# save your trained model checkpoint to wandb
os.environ["WANDB_LOG_MODEL"]="checkpoint"

# turn off watch to log faster
os.environ["WANDB_WATCH"]="false"

In [5]:
# pip install transformers
# from transformers import AutoModelForCausalLM, AutoTokenizer
# checkpoint = "HuggingFaceTB/SmolLM-1.7B-Instruct"

# device = "cuda" # for GPU usage or "cpu" for CPU usage
# tokenizer = AutoTokenizer.from_pretrained(checkpoint)
# # for multiple GPUs install accelerate and do `model = AutoModelForCausalLM.from_pretrained(checkpoint, device_map="auto")`
# model = AutoModelForCausalLM.from_pretrained(checkpoint).to(device)

# messages = [{"role": "user", "content": "What is the capital of France."}]
# input_text=tokenizer.apply_chat_template(messages, tokenize=False)
# print(input_text)
# inputs = tokenizer.encode(input_text, return_tensors="pt").to(device)
# outputs = model.generate(inputs, max_new_tokens=50, temperature=0.2, top_p=0.9, do_sample=True)
# print(tokenizer.decode(outputs[0]))

In [6]:
!pip install -q peft bitsandbytes trl evaluate accelerate transformers

In [7]:
import torch
import datasets
from datasets import load_dataset, Dataset
from trl import DataCollatorForCompletionOnlyLM, SFTConfig, SFTTrainer

In [8]:
model_id = "HuggingFaceTB/SmolLM-1.7B-Instruct"
dataset_id = "NingLab/ECInstruct"
device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

cuda


In [9]:
from transformers import AutoTokenizer
from transformers import AutoModelForSequenceClassification
tokenizer = AutoTokenizer.from_pretrained(model_id)
model=AutoModelForSequenceClassification.from_pretrained(model_id, num_labels=4).to(device)
print(model)

tokenizer_config.json:   0%|          | 0.00/3.59k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/801k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/466k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.10M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/655 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/738 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.42G [00:00<?, ?B/s]

Some weights of LlamaForSequenceClassification were not initialized from the model checkpoint at HuggingFaceTB/SmolLM-1.7B-Instruct and are newly initialized: ['score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


LlamaForSequenceClassification(
  (model): LlamaModel(
    (embed_tokens): Embedding(49152, 2048, padding_idx=2)
    (layers): ModuleList(
      (0-23): 24 x LlamaDecoderLayer(
        (self_attn): LlamaSdpaAttention(
          (q_proj): Linear(in_features=2048, out_features=2048, bias=False)
          (k_proj): Linear(in_features=2048, out_features=2048, bias=False)
          (v_proj): Linear(in_features=2048, out_features=2048, bias=False)
          (o_proj): Linear(in_features=2048, out_features=2048, bias=False)
          (rotary_emb): LlamaRotaryEmbedding()
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear(in_features=2048, out_features=8192, bias=False)
          (up_proj): Linear(in_features=2048, out_features=8192, bias=False)
          (down_proj): Linear(in_features=8192, out_features=2048, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((2048,), eps=1e

In [10]:
dataset = load_dataset(dataset_id)


README.md:   0%|          | 0.00/3.29k [00:00<?, ?B/s]

dataset.json:   0%|          | 0.00/529M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/263562 [00:00<?, ? examples/s]

In [11]:
dataset

DatasetDict({
    train: Dataset({
        features: ['output', 'input', 'options', 'task', 'setting', 'split', 'few_shot_example', 'instruction'],
        num_rows: 263562
    })
})

In [12]:
dataset["train"][0]

{'output': 'A: The product is relevant to the query, and satisfies all the query specifications.',
 'input': '{"query": "fathers christmas gift", "product title": "RAK Magnetic Wristband - Men & Women\'s Tool Bracelet with 10 Strong Magnets to Hold Screws, Nails and Drilling Bits - Gift Ideas for Dad, Husband, Handyman or Handy Woman"}',
 'options': '["A: The product is relevant to the query, and satisfies all the query specifications.", "B: The product is somewhat relevant. It fails to fulfill some aspects of the query but the product can be used as a functional substitute.", "C: The product does not fulfill the query, but could be used in combination with a product exactly matching the query.", "D: The product is irrelevant to the query."]',
 'task': 'Multiclass_Product_Classification',
 'setting': 'IND_Single_Instruction',
 'split': 'train',
 'few_shot_example': None,
 'instruction': 'What is the relevance between the query and the product title below? Answer from one of the options

In [13]:
sentiment_analysis_dataset = dataset.filter(lambda x: x['task'] == 'Multiclass_Product_Classification')
sentiment_analysis_dataset

Filter:   0%|          | 0/263562 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['output', 'input', 'options', 'task', 'setting', 'split', 'few_shot_example', 'instruction'],
        num_rows: 26000
    })
})

In [14]:
# label_map = {
#     "very negative": 0,
#     "negative": 1,
#     "neutral": 2,
#     "positive": 3,
#     "very positive": 4,
# }

label_map = {
    "The product is relevant to the query, and satisfies all the query specifications.": 0, 
    "The product is somewhat relevant. It fails to fulfill some aspects of the query but the product can be used as a functional substitute.":1,
    "The product does not fulfill the query, but could be used in combination with a product exactly matching the query.":2,
    "The product is irrelevant to the query.":3
}

In [15]:
def preprocess_function(examples):
    # Tokenize the input text
    inputs = tokenizer(examples["input"], truncation=True, padding="max_length", max_length=512)
    # Clean the output and map the cleaned label to an integer using the label_map
    cleaned_labels = [clean_output_label(label) for label in examples["output"]]
    labels = [label_map[label] for label in cleaned_labels]
    # Add labels to the tokenized inputs
    inputs["labels"] = labels
    return inputs

In [16]:
import re
def clean_output_label(output):
    """Remove any 'A:', 'B:', 'C:', etc. prefix from the output string using a regular expression."""
    # Use a regular expression to remove any prefix in the form 'A:', 'B:', 'C:', etc.
    cleaned_label = re.sub(r'^[A-E]:\s*', '', output).strip()
    return cleaned_label


In [17]:
def prepare_datasets(dataset):
    """Format, split, and tokenize the dataset for training and evaluation."""
    # Filter out any columns that are not 'input' or 'output'
    keys = list(dataset["train"].features.keys())
    cols_to_remove = [key for key in keys if key not in ["output", "input"]]
    formatted_dataset = dataset.remove_columns(cols_to_remove)

    # Split into train and evaluation datasets
    train_dataset = dataset.filter(lambda x: x['split'] == 'train')
    train_dataset = train_dataset.remove_columns(cols_to_remove)['train']
    eval_dataset = dataset.filter(lambda x: x['split'] == 'test')
    eval_dataset = eval_dataset.remove_columns(cols_to_remove)['train']

# USE FOR DEBUGGING
    
#         # Step 1: Split the dataset to get exactly 5 samples for evaluation
#     eval_dataset = formatted_dataset["train"].shuffle(seed=42).select(range(5))

#     # Step 2: Remove these 5 samples from the original dataset and then split the rest
#     remaining_dataset = formatted_dataset["train"].shuffle(seed=42).select(range(5, len(formatted_dataset["train"])))

#     # Step 3: Now split the remaining dataset to have 0.01% for training
#     train_dataset = remaining_dataset.train_test_split(train_size=30)["train"]

    # Tokenize and prepare the datasets
    tokenized_train_dataset = train_dataset.map(preprocess_function, batched=True)
    tokenized_eval_dataset = eval_dataset.map(preprocess_function, batched=True)

    return tokenized_train_dataset, tokenized_eval_dataset

In [18]:
train_dataset, eval_dataset = prepare_datasets(sentiment_analysis_dataset)

Filter:   0%|          | 0/26000 [00:00<?, ? examples/s]

Filter:   0%|          | 0/26000 [00:00<?, ? examples/s]

Map:   0%|          | 0/20000 [00:00<?, ? examples/s]

Map:   0%|          | 0/4000 [00:00<?, ? examples/s]

In [19]:
print(f"{train_dataset = }")
print(f"{eval_dataset = }")

train_dataset = Dataset({
    features: ['output', 'input', 'input_ids', 'attention_mask', 'labels'],
    num_rows: 20000
})
eval_dataset = Dataset({
    features: ['output', 'input', 'input_ids', 'attention_mask', 'labels'],
    num_rows: 4000
})


In [20]:
#train_dataset[0]
# basically now the dataset has 3 extra columns: input_ids, attention_mask, labels

In [21]:
#eval_dataset[0]

In [22]:
from peft import LoraConfig, get_peft_model
lora_config = LoraConfig(
    r=16,                # Rank for the low-rank update matrices
    lora_alpha=32,        # Scaling factor
    lora_dropout=0.1,     # Dropout to prevent overfitting
    bias="none",          # No bias in LoRA layers
)

In [23]:
model = get_peft_model(model, lora_config)

In [24]:
from transformers import TrainingArguments, Trainer

In [25]:
from accelerate import Accelerator
accelerator = Accelerator(mixed_precision="fp16")  # Enable mixed precision

# Prepare model, datasets, and tokenizer for accelerator
model, train_dataset, eval_dataset = accelerator.prepare(
    model, train_dataset, eval_dataset
)

/opt/conda/lib/python3.10/site-packages/accelerate/accelerator.py:494: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(**kwargs)


In [26]:
batch_size  = 4
grad_steps = 2
steps = len(train_dataset) // (batch_size * grad_steps)

In [27]:
training_args = TrainingArguments(
    fp16=True,
    report_to='wandb',
    output_dir="./peft_lora_output",
    num_train_epochs=2,
    learning_rate = 5e-5,
    
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    gradient_accumulation_steps=grad_steps,
    eval_strategy="steps",        # `evaluation_strategy` is deprecated
    save_strategy="steps",
    eval_steps=steps,
    save_total_limit = 2, 
    logging_strategy='steps',
    logging_steps=20,
    greater_is_better=True,
    run_name = 'classification_run-2'
    
)


trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    tokenizer=tokenizer
)


/opt/conda/lib/python3.10/site-packages/accelerate/accelerator.py:494: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(**kwargs)


In [28]:
trainer.train()
wandb.finish()

wandb: WARNING Changes to your `wandb` environment variables will be ignored because your `wandb` session has already started. For more information on how to modify your settings with `wandb.init()` arguments, please refer to https://wandb.me/wandb-init.
wandb: Currently logged in as: dyotak (dyotak-vesit). Use `wandb login --relogin` to force relogin
wandb: Tracking run with wandb version 0.18.3
wandb: Run data is saved locally in /kaggle/working/wandb/run-20241006_190739-3w8p0gtg
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run classification_run-2
wandb: ⭐️ View project at https://wandb.ai/dyotak-vesit/MajorProject-try-1-multiclass
wandb: 🚀 View run at https://wandb.ai/dyotak-vesit/MajorProject-try-1-multiclass/runs/3w8p0gtg


Step,Training Loss,Validation Loss
2500,0.844800,No log
5000,0.613000,No log


wandb: Adding directory to artifact (./peft_lora_output/checkpoint-500)... Done. 0.1s
wandb: Adding directory to artifact (./peft_lora_output/checkpoint-1000)... Done. 0.1s
wandb: Adding directory to artifact (./peft_lora_output/checkpoint-1500)... Done. 0.1s
wandb: Adding directory to artifact (./peft_lora_output/checkpoint-2000)... Done. 0.1s
wandb: Adding directory to artifact (./peft_lora_output/checkpoint-2500)... Done. 0.1s
wandb: Adding directory to artifact (./peft_lora_output/checkpoint-3000)... Done. 0.1s
wandb: Adding directory to artifact (./peft_lora_output/checkpoint-3500)... Done. 0.1s
wandb: Adding directory to artifact (./peft_lora_output/checkpoint-4000)... Done. 0.1s
wandb: Adding directory to artifact (./peft_lora_output/checkpoint-4500)... Done. 0.2s
wandb: Adding directory to artifact (./peft_lora_output/checkpoint-5000)... Done. 0.1s
/opt/conda/lib/python3.10/site-packages/accelerate/accelerator.py:494: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is depre

In [29]:
torch.cuda.empty_cache()

In [30]:
unmerge = model.merge_and_unload()

In [31]:
unmerge.save_pretrained("unmerged_model")